# 04 — Synchronisation

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- utiliser `Semaphore` et `BoundedSemaphore` pour limiter l'accès concurrent ;
- synchroniser des threads avec `Barrier` ;
- implémenter le pattern **producteur/consommateur** avec `queue.Queue` ;
- utiliser `Condition` pour des notifications fines entre threads ;
- combiner ces primitives dans des patterns réels.

## Prérequis — ce que vous connaissez déjà

Ce notebook s'adresse à un développeur Python **confirmé**. Vous maîtrisez déjà :

- `threading.Thread`, `Lock`, `RLock`, `Event` ;
- le GIL et son impact ;
- `multiprocessing.Process`, `Pool`, `Queue`, `shared_memory` ;
- `concurrent.futures` : `ThreadPoolExecutor`, `ProcessPoolExecutor`, `Future`, `as_completed`.

Notions que nous allons **introduire** ici :

- `Semaphore`, `BoundedSemaphore` ;
- `Barrier` ;
- `queue.Queue` pour le pattern producteur/consommateur ;
- `Condition` variable.

## Plan

1. Rappel : les primitives déjà vues
2. `Semaphore` — contrôler le degré de parallélisme
3. `BoundedSemaphore` — sémaphore borné
4. `Barrier` — rendez-vous entre threads
5. `queue.Queue` — producteur/consommateur
6. `Condition` — notification conditionnelle
7. Pattern : pool de connexions
8. Synthèse
9. Exercices

---

## 1. Rappel : les primitives déjà vues

| Primitive | Rôle |
|---|---|
| `Lock` | Exclusion mutuelle (1 thread à la fois) |
| `RLock` | Lock réentrant (même thread peut ré-acquérir) |
| `Event` | Drapeau booléen (signal on/off) |

Ce notebook introduit les primitives **restantes** : `Semaphore`, `Barrier`, `Condition` et la `queue.Queue`.

---

## 2. `Semaphore` — contrôler le degré de parallélisme

Un `Semaphore(n)` permet à **au plus `n` threads** d'entrer dans la section critique simultanément. C'est un Lock généralisé : `Lock` = `Semaphore(1)`.

In [ ]:
import threading
import time

# Maximum 3 threads simultanés
semaphore = threading.Semaphore(3)

def worker(nom: str) -> None:
    with semaphore:
        print(f"[{nom}] entre (actifs : {3 - semaphore._value}/3)")
        time.sleep(0.5)
        print(f"[{nom}] sort")

threads = [threading.Thread(target=worker, args=(f"T-{i}",)) for i in range(8)]
for t in threads: t.start()
for t in threads: t.join()
print("Tous terminés.")

### Cas d'usage typique : rate limiting

Limiter le nombre de requêtes simultanées vers une API externe.

In [ ]:
import threading
import time

api_limiter = threading.Semaphore(2)  # max 2 appels en parallèle

def appel_api(endpoint: str) -> str:
    with api_limiter:
        print(f"→ {endpoint}")
        time.sleep(0.3)  # simule l'appel
        return f"{endpoint} : 200 OK"

threads = [threading.Thread(target=appel_api, args=(f"/users/{i}",)) for i in range(6)]
for t in threads: t.start()
for t in threads: t.join()

---

## 3. `BoundedSemaphore` — sémaphore borné

Un `BoundedSemaphore` interdit de `release()` **plus de fois** qu'il n'a été `acquire()`. Cela protège contre les bugs de programmation.

In [ ]:
import threading

# Semaphore normal : release() excédentaire passe
sem = threading.Semaphore(2)
sem.release()  # passe à 3 — pas d'erreur
print(f"Semaphore._value après release excédentaire : {sem._value}")

# BoundedSemaphore : release() excédentaire → ValueError
bsem = threading.BoundedSemaphore(2)
try:
    bsem.release()  # pas d'acquire préalable
except ValueError as e:
    print(f"BoundedSemaphore : {e}")

**Bonne pratique :** préférer `BoundedSemaphore` pour détecter les bugs plus tôt.

---

## 4. `Barrier` — rendez-vous entre threads

Une `Barrier(n)` bloque chaque thread appelant `wait()` jusqu'à ce que **exactement `n` threads** aient tous appelé `wait()`. Puis ils sont tous relâchés.

In [ ]:
import threading
import time
import random

barrier = threading.Barrier(4)

def etape(nom: str) -> None:
    # Phase 1 : initialisation (durées différentes)
    init_time = random.uniform(0.1, 0.5)
    time.sleep(init_time)
    print(f"[{nom}] Prêt après {init_time:.2f}s")

    barrier.wait()  # tous doivent arriver ici

    # Phase 2 : exécution synchronisée
    print(f"[{nom}] GO !")

threads = [threading.Thread(target=etape, args=(f"T-{i}",)) for i in range(4)]
for t in threads: t.start()
for t in threads: t.join()

### Barrier avec action

On peut passer une fonction `action` exécutée par **un seul** thread quand la barrière est franchie.

In [ ]:
import threading
import time

def on_barrier():
    print("=== Tous les workers sont prêts, lancement ! ===")

barrier = threading.Barrier(3, action=on_barrier)

def worker(nom: str) -> None:
    time.sleep(0.2)
    print(f"[{nom}] arrivé à la barrière")
    barrier.wait()
    print(f"[{nom}] travaille")

threads = [threading.Thread(target=worker, args=(f"W-{i}",)) for i in range(3)]
for t in threads: t.start()
for t in threads: t.join()

---

## 5. `queue.Queue` — producteur/consommateur

Le module `queue` fournit des files **thread-safe** sans avoir besoin de verrous manuels. C'est l'outil de choix pour le pattern **producteur/consommateur**.

### 5.1. `Queue` — FIFO

In [ ]:
import queue
import threading
import time

q = queue.Queue(maxsize=5)  # tampon borné

def producteur() -> None:
    for i in range(10):
        q.put(f"item-{i}")
        print(f"[P] produit item-{i}")
        time.sleep(0.05)
    q.put(None)  # sentinel

def consommateur() -> None:
    while True:
        item = q.get()
        if item is None:
            break
        print(f"[C] consommé {item}")
        q.task_done()

t_prod = threading.Thread(target=producteur)
t_cons = threading.Thread(target=consommateur)
t_cons.start()
t_prod.start()
t_prod.join()
t_cons.join()
print("Pipeline terminé.")

### 5.2. Plusieurs producteurs, plusieurs consommateurs

In [ ]:
import queue
import threading
import time

SENTINEL = object()
q = queue.Queue(maxsize=10)

def producteur(nom: str, n: int) -> None:
    for i in range(n):
        q.put(f"{nom}-{i}")
        time.sleep(0.02)

def consommateur(nom: str) -> None:
    while True:
        item = q.get()
        if item is SENTINEL:
            q.task_done()
            break
        print(f"[{nom}] {item}")
        q.task_done()

nb_consumers = 3
producers = [
    threading.Thread(target=producteur, args=(f"P{i}", 5))
    for i in range(2)
]
consumers = [
    threading.Thread(target=consommateur, args=(f"C{i}",))
    for i in range(nb_consumers)
]

for c in consumers: c.start()
for p in producers: p.start()
for p in producers: p.join()

# Envoyer les sentinels pour arrêter les consommateurs
for _ in range(nb_consumers):
    q.put(SENTINEL)

for c in consumers: c.join()
print("Terminé.")

### 5.3. `PriorityQueue` et `LifoQueue`

In [ ]:
import queue

# PriorityQueue : les éléments sortent par priorité (plus petit d'abord)
pq = queue.PriorityQueue()
pq.put((3, "basse"))
pq.put((1, "haute"))
pq.put((2, "moyenne"))

while not pq.empty():
    print(pq.get())

In [ ]:
import queue

# LifoQueue : Last In, First Out (pile)
lq = queue.LifoQueue()
for i in range(5):
    lq.put(i)

while not lq.empty():
    print(lq.get(), end=" ")
print()

---

## 6. `Condition` — notification conditionnelle

Une `Condition` combine un `Lock` avec un mécanisme de **notification** (`wait()` / `notify()` / `notify_all()`). Elle permet à un thread de dire « je suis prêt » ou « une donnée est disponible ».

In [ ]:
import threading
import time

condition = threading.Condition()
donnee_prete = False
donnee = None

def producteur() -> None:
    global donnee_prete, donnee
    time.sleep(1)  # simule la production
    with condition:
        donnee = "résultat important"
        donnee_prete = True
        condition.notify_all()  # réveille tous les waiters
        print("[P] Donnée prête, notification envoyée")

def consommateur(nom: str) -> None:
    with condition:
        condition.wait_for(lambda: donnee_prete)  # attend la condition
        print(f"[{nom}] Reçu : {donnee}")

c1 = threading.Thread(target=consommateur, args=("C1",))
c2 = threading.Thread(target=consommateur, args=("C2",))
p = threading.Thread(target=producteur)

c1.start(); c2.start()
p.start()
c1.join(); c2.join(); p.join()

### `wait_for()` vs `wait()`

- `wait()` attend simplement un `notify()`. Attention aux **spurious wakeups** !
- `wait_for(predicate)` boucle automatiquement jusqu'à ce que le prédicat soit vrai. **Préférez toujours `wait_for()`**.

In [ ]:
import threading

# Démonstration de wait_for avec un buffer borné
condition = threading.Condition()
buffer: list[int] = []
MAX_SIZE = 3

def producteur_conditionnel() -> None:
    for i in range(6):
        with condition:
            condition.wait_for(lambda: len(buffer) < MAX_SIZE)
            buffer.append(i)
            print(f"[P] Ajouté {i}, buffer = {buffer}")
            condition.notify()

def consommateur_conditionnel() -> None:
    for _ in range(6):
        with condition:
            condition.wait_for(lambda: len(buffer) > 0)
            item = buffer.pop(0)
            print(f"[C] Retiré {item}, buffer = {buffer}")
            condition.notify()

t1 = threading.Thread(target=producteur_conditionnel)
t2 = threading.Thread(target=consommateur_conditionnel)
t1.start(); t2.start()
t1.join(); t2.join()

---

## 7. Pattern : pool de connexions

Un exemple concret qui combine `Semaphore` et une `Queue` pour créer un pool de connexions réutilisables.

In [ ]:
import threading
import queue
import time

class ConnectionPool:
    """Pool de connexions thread-safe."""
    def __init__(self, size: int) -> None:
        self._pool: queue.Queue[int] = queue.Queue(maxsize=size)
        for i in range(size):
            self._pool.put(i)  # IDs de connexion simulés

    def acquire(self, timeout: float | None = None) -> int:
        try:
            return self._pool.get(timeout=timeout)
        except queue.Empty:
            raise TimeoutError("Pas de connexion disponible")

    def release(self, conn_id: int) -> None:
        self._pool.put(conn_id)

pool = ConnectionPool(3)

def worker(nom: str) -> None:
    conn = pool.acquire(timeout=2)
    print(f"[{nom}] Connexion {conn} acquise")
    time.sleep(0.3)  # utilise la connexion
    pool.release(conn)
    print(f"[{nom}] Connexion {conn} relâchée")

threads = [threading.Thread(target=worker, args=(f"W-{i}",)) for i in range(6)]
for t in threads: t.start()
for t in threads: t.join()
print("Pool de connexions : tout est relâché.")

---

## 8. Synthèse

| Primitive | Rôle | Cas d'usage |
|---|---|---|
| `Semaphore(n)` | Max `n` threads simultanés | Rate limiting, pool de ressources |
| `BoundedSemaphore(n)` | Idem + détection de bugs | Préférer à `Semaphore` |
| `Barrier(n)` | Rendez-vous de `n` threads | Synchronisation de phases |
| `queue.Queue` | File FIFO thread-safe | Producteur/consommateur |
| `queue.PriorityQueue` | File à priorité | Tâches ordonnées |
| `Condition` | Lock + wait/notify | Notification conditionnelle |

**Quand utiliser quoi ?**

- Limiter l'accès à `n` ressources → `Semaphore`
- Attendre que tous les threads soient prêts → `Barrier`
- Passer des données entre threads → `queue.Queue`
- Signaler un changement d'état → `Condition`

---

## 9. Exercices

### Exercice 1 — Rate limiter *(facile)*

Simuler 10 appels API avec un `Semaphore(3)`. Chaque appel dure 0.3s. Mesurer le temps total et vérifier qu'il est proche de `ceil(10/3) * 0.3 = 1.2s`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Synchronisation", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
import threading
import time

sem = threading.Semaphore(3)

def appel_api(i: int) -> None:
    with sem:
        time.sleep(0.3)
        print(f"API call {i} done")

start = time.perf_counter()
threads = [threading.Thread(target=appel_api, args=(i,)) for i in range(10)]
for t in threads: t.start()
for t in threads: t.join()
elapsed = time.perf_counter() - start
print(f"Temps total : {elapsed:.2f}s (attendu ~1.2s)")
```

</details>

### Exercice 2 — Pipeline à 3 étages *(moyen)*

Implémenter un pipeline avec 3 `queue.Queue` :

1. **Producteur** : génère des nombres de 0 à 19.
2. **Transformateur** : lit, calcule le carré, écrit dans la queue suivante.
3. **Afficheur** : lit et affiche.

Chaque étage est un thread séparé. Utiliser des sentinels pour l'arrêt.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Synchronisation", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import queue
import threading

SENTINEL = None
q1 = queue.Queue()
q2 = queue.Queue()

def producteur() -> None:
    for i in range(20):
        q1.put(i)
    q1.put(SENTINEL)

def transformateur() -> None:
    while True:
        item = q1.get()
        if item is SENTINEL:
            q2.put(SENTINEL)
            break
        q2.put(item * item)

def afficheur() -> None:
    while True:
        item = q2.get()
        if item is SENTINEL:
            break
        print(f"  {item}")

threads = [
    threading.Thread(target=producteur),
    threading.Thread(target=transformateur),
    threading.Thread(target=afficheur),
]
for t in threads: t.start()
for t in threads: t.join()
print("Pipeline terminé.")
```

</details>

### Exercice 3 — Barrier pour benchmark *(moyen)*

4 threads exécutent chacun un calcul CPU. Utiliser une `Barrier(4)` pour qu'ils démarrent **exactement en même temps**, puis une seconde `Barrier(4)` pour attendre qu'ils aient tous fini avant d'afficher les temps.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Synchronisation", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
import threading
import time

start_barrier = threading.Barrier(4)
end_barrier = threading.Barrier(4)
resultats: dict[str, float] = {}
lock = threading.Lock()

def benchmark(nom: str, n: int) -> None:
    start_barrier.wait()  # tous démarrent ensemble
    t0 = time.perf_counter()
    sum(range(n))
    elapsed = time.perf_counter() - t0
    with lock:
        resultats[nom] = elapsed
    end_barrier.wait()  # attendent tous d'avoir fini

threads = [
    threading.Thread(target=benchmark, args=(f"T-{i}", 5_000_000))
    for i in range(4)
]
for t in threads: t.start()
for t in threads: t.join()

for nom, temps in sorted(resultats.items()):
    print(f"  {nom}: {temps:.3f}s")
```

</details>

### Exercice 4 — Lecteurs/Écrivain *(difficile)*

Implémenter le pattern **lecteurs/écrivain** avec une `Condition` :

- Plusieurs lecteurs peuvent lire simultanément.
- Un écrivain a un accès exclusif (pas de lecteurs ni d'autres écrivains).
- Les écrivains ont priorité sur les lecteurs.

Tester avec 5 lecteurs et 2 écrivains.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Synchronisation", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
import threading
import time
import random

class ReadWriteLock:
    def __init__(self) -> None:
        self._cond = threading.Condition()
        self._readers = 0
        self._writer = False
        self._waiting_writers = 0

    def acquire_read(self) -> None:
        with self._cond:
            self._cond.wait_for(
                lambda: not self._writer and self._waiting_writers == 0
            )
            self._readers += 1

    def release_read(self) -> None:
        with self._cond:
            self._readers -= 1
            if self._readers == 0:
                self._cond.notify_all()

    def acquire_write(self) -> None:
        with self._cond:
            self._waiting_writers += 1
            self._cond.wait_for(
                lambda: not self._writer and self._readers == 0
            )
            self._waiting_writers -= 1
            self._writer = True

    def release_write(self) -> None:
        with self._cond:
            self._writer = False
            self._cond.notify_all()

rw = ReadWriteLock()
data = {"value": 0}

def lecteur(nom: str) -> None:
    for _ in range(3):
        rw.acquire_read()
        print(f"[{nom}] Lit : {data['value']}")
        time.sleep(random.uniform(0.05, 0.1))
        rw.release_read()

def ecrivain(nom: str) -> None:
    for i in range(2):
        rw.acquire_write()
        data["value"] += 1
        print(f"[{nom}] Écrit : {data['value']}")
        time.sleep(0.1)
        rw.release_write()

threads = (
    [threading.Thread(target=lecteur, args=(f"L{i}",)) for i in range(5)]
    + [threading.Thread(target=ecrivain, args=(f"W{i}",)) for i in range(2)]
)
for t in threads: t.start()
for t in threads: t.join()
print(f"Valeur finale : {data['value']}")
```

</details>

---

## Ressources

- [docs Python — `threading`](https://docs.python.org/3/library/threading.html)
- [docs Python — `queue`](https://docs.python.org/3/library/queue.html)
- *The Little Book of Semaphores* (Allen B. Downey) — [gratuit en ligne](https://greenteapress.com/wp/semaphores/)
- *Programming with POSIX Threads* (D. Butenhof)